<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4K0B_Primary_Event_Component_Bootstrap_Support_Materialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GES Stage 6C — Cell 6C-4K0B
## Primary Event-Component Bootstrap Support Materialization

Run the single code cell below in Google Colab. This serializes the already-completed Cell 6C-3E1 analysis and does not start Experiment 2.


In [1]:
# ==================================================================================================
# STAGE 6C — CELL 6C-4K0B
# PRIMARY EVENT-COMPONENT BOOTSTRAP SUPPORT MATERIALIZATION
# ==================================================================================================
#
# Purpose
# -------
# The original Cells 6C-3E0/6C-3E1 completed the frozen three-component analysis in memory,
# but did not independently serialize the full bootstrap-support package. This cell reconstructs
# that completed analysis exactly and writes versioned, checksum-protected support artifacts for
# the final integrated Stage 6C freeze.
#
# Scientific boundary
# -------------------
# - Uses only the immutable 66,636-row Stage 6B primary-evaluable cohort.
# - Uses the three already-frozen Stage 5 event-component fields.
# - Uses the nine already-frozen Stage 6B risk scores.
# - Uses the original Cell 6C-3E1 bootstrap implementation:
#       numpy.random.default_rng(42)
#       multinomial row-count bootstrap
#       2,000 attempts
#       batch size 50
#       identical samples across all three components and all nine scores
# - Does not refit, recalibrate, retune, redefine, optimize, or modify any scientific input.
# - Experiment 2 remains unstarted and unauthorized.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import scipy
from scipy.sparse import csr_matrix, vstack
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. LOCKED INPUTS, EXPECTATIONS, AND OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_NAME = (
    "GES_Stage6C_Cell_6C_4K0B_Primary_Event_Component_"
    "Bootstrap_Support_Materialization.ipynb"
)

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

SOURCE = ROOT / (
    "data_processed/stage6_temporal_validation/"
    "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
SOURCE_SHA256 = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4k0a_frozen_threshold_bootstrap_support_materialization_v1/"
    "stage6c_frozen_threshold_bootstrap_support_manifest_v1.json"
)
PRIOR_MANIFEST_SHA256 = "959c6deebb69956dc19b6427f115f0f7644dbd70d426d1914ad148191417c0b7"

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

SEED = 42
N_BOOT = 2_000
BOOTSTRAP_BATCH_SIZE = 50
RISK_FRACTIONS = [0.05, 0.10, 0.20]

PRIMARY_OUTCOME = "primary_future_instability"

COMPONENTS = {
    "material_group_change": {
        "column": "event_material_clinical_group_change",
        "display": "Material clinical-group change",
        "expected_events": 1_405,
    },
    "new_unresolved_conflict": {
        "column": "event_new_unresolved_conflict_at_t1",
        "display": "New unresolved conflict",
        "expected_events": 4_789,
    },
    "material_prior_conflict_resolution": {
        "column": "event_prior_conflict_resolved_to_material_group",
        "display": "Material prior-conflict resolution",
        "expected_events": 297,
    },
}

MODELS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display": "Additive risk",
    },
}

PRINCIPAL_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]
SECONDARY_COMPARATORS = ["conflict", "recency", "submitter", "entropy", "additive"]

COMPONENT_KEYS = list(COMPONENTS)
MODEL_KEYS = list(MODELS)

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4k0b_primary_event_component_bootstrap_support_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4k0b_primary_event_component_bootstrap_support_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4k0b_primary_event_component_bootstrap_support_materialization_v1"
)

for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "accounting": TABLE_DIR / "stage6c_primary_event_component_accounting_v1.csv",
    "points": TABLE_DIR / "stage6c_primary_event_component_point_estimates_v1.csv",
    "replicates": TABLE_DIR / "stage6c_primary_event_component_bootstrap_replicates_v1.parquet",
    "validity": TABLE_DIR / "stage6c_primary_event_component_bootstrap_validity_v1.csv",
    "intervals": TABLE_DIR / "stage6c_primary_event_component_model_bootstrap_intervals_v1.csv",
    "paired": TABLE_DIR / "stage6c_primary_event_component_paired_bootstrap_inference_v1.csv",
    "enrichment": TABLE_DIR / "stage6c_primary_event_component_enrichment_bootstrap_intervals_v1.csv",
    "historical": TABLE_DIR / "stage6c_primary_event_component_historical_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_primary_event_component_historical_concordance_v1.csv",
    "qc": QC_DIR / "stage6c_4k0b_primary_event_component_bootstrap_support_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_primary_event_component_bootstrap_support_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)

    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(
            native(obj),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    frame.to_parquet(temporary, index=False, compression="zstd", engine="pyarrow")

    if path.exists():
        old = pd.read_parquet(path)
        new = pd.read_parquet(temporary)
        pd.testing.assert_frame_equal(old, new, check_dtype=True, check_exact=True)
        temporary.unlink()
    else:
        os.replace(temporary, path)

    return sha(path)


def sidecar(path: Path) -> Path:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    stable_write_bytes(output, f"{sha(path)}  {path.name}\n".encode("utf-8"))
    return output


def sidecar_hash(path: Path) -> str:
    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        Path(path).read_text(encoding="utf-8"),
    )
    if not matches:
        raise RuntimeError(f"No SHA-256 found in sidecar: {path}")
    return matches[0].lower()


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    return output.exists() and sidecar_hash(output) == sha(path)


def binary_column(frame: pd.DataFrame, column: str) -> np.ndarray:
    if column not in frame:
        raise KeyError(column)
    series = frame[column]

    if pd.api.types.is_bool_dtype(series):
        if series.isna().any():
            raise RuntimeError(f"Binary column contains missing values: {column}")
        return series.to_numpy(dtype=np.int8)

    numeric = pd.to_numeric(series, errors="raise")
    if numeric.isna().any():
        raise RuntimeError(f"Binary column contains missing values: {column}")
    if not set(numeric.unique()).issubset({0, 1, 0.0, 1.0}):
        raise RuntimeError(f"Column is not binary: {column}")
    return numeric.to_numpy(dtype=np.int8)


def ci(values) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    low, high = np.quantile(values, [0.025, 0.975])
    return float(low), float(high)


def sign_p(values) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    lower_tail = (np.count_nonzero(values <= 0.0) + 1) / (len(values) + 1)
    upper_tail = (np.count_nonzero(values >= 0.0) + 1) / (len(values) + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm(values) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    adjusted = np.full(len(values), np.nan, dtype=float)
    valid_positions = np.flatnonzero(np.isfinite(values))

    if len(valid_positions) == 0:
        return adjusted

    valid = values[valid_positions]
    order = np.argsort(valid)
    m = len(valid)
    running_max = 0.0

    for rank, ordered_position in enumerate(order):
        original_position = valid_positions[ordered_position]
        running_max = max(running_max, (m - rank) * valid[ordered_position])
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(lower: float, upper: float, positive: str, negative: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0:
        return positive
    if upper < 0:
        return negative
    return "interval_includes_null"


def score_group_cache(scores: np.ndarray, outcome_arrays: dict[str, np.ndarray]) -> dict:
    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_groups = len(unique_scores)
    row_positions = np.arange(len(scores), dtype=np.int64)

    total_matrix = csr_matrix(
        (
            np.ones(len(scores), dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, len(scores)),
    )

    positive_matrices = []
    for component_key in COMPONENT_KEYS:
        outcome = outcome_arrays[component_key]
        positive_positions = np.flatnonzero(outcome == 1)
        positive_matrices.append(
            csr_matrix(
                (
                    np.ones(len(positive_positions), dtype=np.float64),
                    (group_index[positive_positions], positive_positions),
                ),
                shape=(n_groups, len(scores)),
            )
        )

    return {
        "n_groups": n_groups,
        "total_matrix": total_matrix,
        "positive_stack": vstack(positive_matrices, format="csr"),
    }


def grouped_metric_batch(
    total_group_counts: np.ndarray,
    positive_group_counts: np.ndarray,
    positive_totals: np.ndarray,
    sample_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    total_group_counts = np.asarray(total_group_counts, dtype=np.float64)
    positive_group_counts = np.asarray(positive_group_counts, dtype=np.float64)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)
    sample_totals = np.asarray(sample_totals, dtype=np.float64)

    negative_totals = sample_totals - positive_totals
    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    positive_desc = positive_group_counts[::-1, :]
    total_desc = total_group_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.divide(
        cumulative_positive,
        cumulative_total,
        out=np.zeros_like(cumulative_positive),
        where=cumulative_total > 0.0,
    )

    ap = np.full(len(positive_totals), np.nan, dtype=np.float64)
    ap_numerator = np.sum(positive_desc * precision, axis=0)
    np.divide(ap_numerator, positive_totals, out=ap, where=valid)

    negative_group_counts = total_group_counts - positive_group_counts
    negatives_before = np.cumsum(negative_group_counts, axis=0) - negative_group_counts
    auc_numerator = np.sum(
        positive_group_counts
        * (negatives_before + 0.5 * negative_group_counts),
        axis=0,
    )

    auc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        auc_numerator,
        positive_totals * negative_totals,
        out=auc,
        where=valid,
    )

    return ap, auc


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STRUCTURAL PREFLIGHT
# --------------------------------------------------------------------------------------------------

for required in (
    SOURCE,
    SOURCE.with_name(SOURCE.name + ".sha256"),
    PRIOR_MANIFEST,
    PRIOR_MANIFEST.with_name(PRIOR_MANIFEST.name + ".sha256"),
):
    if not required.exists():
        raise FileNotFoundError(required)

if sha(SOURCE) != SOURCE_SHA256 or not sidecar_ok(SOURCE):
    raise RuntimeError("Stage 6B source verification failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256 or not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Prior Cell 6C-4K0A manifest verification failed.")

prior_manifest_readback = json.loads(PRIOR_MANIFEST.read_text(encoding="utf-8"))
if prior_manifest_readback.get("decision") != (
    "PASS_STAGE6C_FROZEN_THRESHOLD_BOOTSTRAP_SUPPORT_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
):
    raise RuntimeError("Prior Cell 6C-4K0A decision mismatch.")

metadata = pq.ParquetFile(SOURCE).metadata
if (metadata.num_rows, metadata.num_columns) != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise RuntimeError(
        f"Unexpected Stage 6B dimensions: {(metadata.num_rows, metadata.num_columns)}"
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD AND REVERIFY THE FROZEN THREE-COMPONENT DECOMPOSITION
# --------------------------------------------------------------------------------------------------

required_columns = [
    PRIMARY_OUTCOME,
    *[spec["column"] for spec in COMPONENTS.values()],
    *[spec["column"] for spec in MODELS.values()],
]

frame = pd.read_parquet(SOURCE, columns=required_columns)
if frame.shape[0] != EXPECTED_ROWS:
    raise RuntimeError("Loaded Stage 6B row count mismatch.")

primary = binary_column(frame, PRIMARY_OUTCOME)
if (
    int(primary.sum()),
    int(len(primary) - primary.sum()),
) != (
    EXPECTED_PRIMARY_EVENTS,
    EXPECTED_PRIMARY_NEGATIVES,
):
    raise RuntimeError("Frozen primary-outcome accounting mismatch.")

outcomes = {
    key: binary_column(frame, specification["column"])
    for key, specification in COMPONENTS.items()
}

for key, outcome in outcomes.items():
    observed = int(outcome.sum())
    expected = COMPONENTS[key]["expected_events"]
    if observed != expected:
        raise RuntimeError(f"{key} events={observed}; expected={expected}")

material = outcomes["material_group_change"].astype(bool)
new_conflict = outcomes["new_unresolved_conflict"].astype(bool)
prior_resolution = outcomes["material_prior_conflict_resolution"].astype(bool)

overlap_material_new = int(np.count_nonzero(material & new_conflict))
overlap_material_resolution = int(np.count_nonzero(material & prior_resolution))
overlap_new_resolution = int(np.count_nonzero(new_conflict & prior_resolution))
triple_overlap = int(np.count_nonzero(material & new_conflict & prior_resolution))
component_union = (material | new_conflict | prior_resolution).astype(np.int8)

if overlap_material_new != 6:
    raise RuntimeError("Material/new-conflict overlap changed.")
if overlap_material_resolution != 0 or overlap_new_resolution != 0:
    raise RuntimeError("Unexpected event-component pairwise overlap.")
if triple_overlap != 0:
    raise RuntimeError("Unexpected event-component triple overlap.")
if not np.array_equal(component_union, primary):
    raise RuntimeError("Three-component OR no longer reconstructs the primary outcome.")

score_arrays = {}
for model_key, specification in MODELS.items():
    values = pd.to_numeric(
        frame[specification["column"]],
        errors="raise",
    ).to_numpy(dtype=float)

    if not np.isfinite(values).all() or values.min() < 0.0 or values.max() > 1.0:
        raise RuntimeError(f"Invalid frozen score: {model_key}")

    score_arrays[model_key] = values

accounting_rows = []
for component_key, specification in COMPONENTS.items():
    outcome = outcomes[component_key]
    accounting_rows.append({
        "analysis_category": "PRIMARY_EVENT_COMPONENT_ACCOUNTING",
        "component_key": component_key,
        "event_component_column": specification["column"],
        "component": specification["display"],
        "rows": EXPECTED_ROWS,
        "component_events": int(outcome.sum()),
        "component_negatives": int(EXPECTED_ROWS - outcome.sum()),
        "component_prevalence": float(outcome.mean()),
        "expected_component_events": specification["expected_events"],
        "component_event_count_matches": int(outcome.sum()) == specification["expected_events"],
    })

accounting = pd.DataFrame(accounting_rows)


# --------------------------------------------------------------------------------------------------
# 5. POINT ESTIMATES AND EXACT GROUPED-METRIC VALIDATION
# --------------------------------------------------------------------------------------------------

caches = {
    model_key: score_group_cache(values, outcomes)
    for model_key, values in score_arrays.items()
}

point_rows = []
metric_validation_rows = []
unit_counts = np.ones((1, EXPECTED_ROWS), dtype=np.float64)
unit_total = np.array([EXPECTED_ROWS], dtype=np.float64)

for model_key, scores in score_arrays.items():
    cache = caches[model_key]
    total_group_counts = np.asarray(
        cache["total_matrix"] @ unit_counts.T,
        dtype=float,
    )
    positive_stacked = np.asarray(
        cache["positive_stack"] @ unit_counts.T,
        dtype=float,
    ).reshape(len(COMPONENT_KEYS), cache["n_groups"], 1)

    for component_index, component_key in enumerate(COMPONENT_KEYS):
        outcome = outcomes[component_key]
        grouped_ap, grouped_auc = grouped_metric_batch(
            total_group_counts,
            positive_stacked[component_index],
            np.array([outcome.sum()], dtype=float),
            unit_total,
        )

        sklearn_ap = float(average_precision_score(outcome, scores))
        sklearn_auc = float(roc_auc_score(outcome, scores))
        ap_difference = abs(float(grouped_ap[0]) - sklearn_ap)
        auc_difference = abs(float(grouped_auc[0]) - sklearn_auc)

        metric_validation_rows.append({
            "component_key": component_key,
            "model_key": model_key,
            "auprc_absolute_difference": ap_difference,
            "auroc_absolute_difference": auc_difference,
        })

        if ap_difference > 1e-12 or auc_difference > 1e-12:
            raise RuntimeError(
                f"Exact grouped metric validation failed: {component_key}/{model_key}"
            )

        prevalence = float(outcome.mean())
        point_rows.append({
            "analysis_category": "PRIMARY_EVENT_COMPONENT_AUPRC_AUROC_POINT_ESTIMATE",
            "component_key": component_key,
            "event_component_column": COMPONENTS[component_key]["column"],
            "component": COMPONENTS[component_key]["display"],
            "model_key": model_key,
            "model": MODELS[model_key]["display"],
            "score_column": MODELS[model_key]["column"],
            "rows": EXPECTED_ROWS,
            "events": int(outcome.sum()),
            "negatives": int(EXPECTED_ROWS - outcome.sum()),
            "prevalence": prevalence,
            "point_auprc": sklearn_ap,
            "auprc_minus_prevalence": sklearn_ap - prevalence,
            "auprc_lift_over_prevalence": sklearn_ap / prevalence,
            "point_auroc": sklearn_auc,
            "auroc_minus_0_50": sklearn_auc - 0.50,
        })

points = pd.DataFrame(point_rows)
point_lookup = points.set_index(["component_key", "model_key"])
metric_validation = pd.DataFrame(metric_validation_rows)


# --------------------------------------------------------------------------------------------------
# 6. EXACT CELL 6C-3E1 BOOTSTRAP STREAM
# --------------------------------------------------------------------------------------------------

n_components = len(COMPONENT_KEYS)
rng = np.random.default_rng(SEED)
outcome_matrix = np.vstack([outcomes[key] for key in COMPONENT_KEYS]).astype(np.int8)

sampled_events = {
    key: np.full(N_BOOT, -1, dtype=np.int32)
    for key in COMPONENT_KEYS
}
validity = {
    key: np.zeros(N_BOOT, dtype=bool)
    for key in COMPONENT_KEYS
}

metric_values = {
    (component_key, model_key, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for component_key in COMPONENT_KEYS
    for model_key in MODEL_KEYS
    for metric in ("auprc", "auroc")
}

enrichment_values = {
    (component_key, fraction, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for component_key in COMPONENT_KEYS
    for fraction in RISK_FRACTIONS
    for metric in (
        "selected_event_rate",
        "risk_ratio_vs_remaining",
        "enrichment_over_prevalence",
    )
}

full_scores = score_arrays["full_ges"]
rank_order = np.argsort(-full_scores, kind="mergesort")
selected_masks = {}

for fraction in RISK_FRACTIONS:
    selected_rows = int(np.ceil(EXPECTED_ROWS * fraction))
    mask = np.zeros(EXPECTED_ROWS, dtype=np.int8)
    mask[rank_order[:selected_rows]] = 1
    selected_masks[fraction] = mask

probabilities = np.full(
    EXPECTED_ROWS,
    1.0 / EXPECTED_ROWS,
    dtype=np.float64,
)
probabilities[-1] = 1.0 - probabilities[:-1].sum()

print(f"Use this Colab notebook file name: {NOTEBOOK_NAME}")
print(
    "\nReconstructing primary event-component bootstrap across "
    f"{EXPECTED_ROWS:,} rows: "
    + ", ".join(
        f"{COMPONENTS[key]['display']}={COMPONENTS[key]['expected_events']:,}"
        for key in COMPONENT_KEYS
    )
)
print("Exact grouped metric validation against scikit-learn: PASS (27/27)")

analysis_start = time.time()

for batch_start in range(0, N_BOOT, BOOTSTRAP_BATCH_SIZE):
    batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOT)
    batch_size = batch_end - batch_start

    # Exact historical Cell 6C-3E1 ordinary row-bootstrap stream.
    count_matrix = rng.multinomial(
        EXPECTED_ROWS,
        probabilities,
        size=batch_size,
    ).astype(np.int32, copy=False)

    if not np.all(count_matrix.sum(axis=1) == EXPECTED_ROWS):
        raise RuntimeError("Bootstrap sample-size preservation failed.")

    count_transpose = count_matrix.T
    event_count_batch = outcome_matrix.astype(np.int64) @ count_transpose
    sample_total_batch = count_matrix.sum(axis=1).astype(np.float64)

    for component_index, component_key in enumerate(COMPONENT_KEYS):
        events_batch = event_count_batch[component_index].astype(np.int32)
        sampled_events[component_key][batch_start:batch_end] = events_batch
        validity[component_key][batch_start:batch_end] = (
            (events_batch > 0) & (events_batch < EXPECTED_ROWS)
        )

    for model_key, cache in caches.items():
        total_group_counts = np.asarray(
            cache["total_matrix"] @ count_transpose,
            dtype=float,
        )
        positive_stacked = np.asarray(
            cache["positive_stack"] @ count_transpose,
            dtype=float,
        ).reshape(n_components, cache["n_groups"], batch_size)

        for component_index, component_key in enumerate(COMPONENT_KEYS):
            aps, aucs = grouped_metric_batch(
                total_group_counts,
                positive_stacked[component_index],
                event_count_batch[component_index],
                sample_total_batch,
            )
            metric_values[
                (component_key, model_key, "auprc")
            ][batch_start:batch_end] = aps
            metric_values[
                (component_key, model_key, "auroc")
            ][batch_start:batch_end] = aucs

    for fraction, selected_mask in selected_masks.items():
        sampled_selected_rows = (
            selected_mask.astype(np.int64) @ count_transpose
        )
        sampled_remaining_rows = EXPECTED_ROWS - sampled_selected_rows

        for component_index, component_key in enumerate(COMPONENT_KEYS):
            outcome = outcome_matrix[component_index].astype(np.int64)
            selected_event_indicator = outcome * selected_mask

            sampled_selected_events = (
                selected_event_indicator @ count_transpose
            )
            sampled_total_events = event_count_batch[component_index]
            sampled_remaining_events = (
                sampled_total_events - sampled_selected_events
            )

            selected_rate = np.divide(
                sampled_selected_events,
                sampled_selected_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_selected_rows > 0,
            )
            remaining_rate = np.divide(
                sampled_remaining_events,
                sampled_remaining_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_remaining_rows > 0,
            )
            prevalence_batch = sampled_total_events / EXPECTED_ROWS

            risk_ratio = np.divide(
                selected_rate,
                remaining_rate,
                out=np.full(batch_size, np.nan, dtype=float),
                where=remaining_rate > 0,
            )
            enrichment = np.divide(
                selected_rate,
                prevalence_batch,
                out=np.full(batch_size, np.nan, dtype=float),
                where=prevalence_batch > 0,
            )

            enrichment_values[
                (component_key, fraction, "selected_event_rate")
            ][batch_start:batch_end] = selected_rate
            enrichment_values[
                (component_key, fraction, "risk_ratio_vs_remaining")
            ][batch_start:batch_end] = risk_ratio
            enrichment_values[
                (component_key, fraction, "enrichment_over_prevalence")
            ][batch_start:batch_end] = enrichment

    if batch_end % 250 == 0:
        valid_text = ", ".join(
            f"{key}={int(validity[key][:batch_end].sum()):,}"
            for key in COMPONENT_KEYS
        )
        print(
            f"  Completed {batch_end:,}/{N_BOOT:,} replicates | "
            f"valid [{valid_text}]"
        )

bootstrap_elapsed = time.time() - analysis_start


# --------------------------------------------------------------------------------------------------
# 7. MATERIALIZE REPLICATES, VALIDITY, INTERVALS, PAIRED INFERENCE, AND ENRICHMENT
# --------------------------------------------------------------------------------------------------

replicate_data = {
    "analysis_category": np.full(
        N_BOOT,
        "PRIMARY_EVENT_COMPONENT_BOOTSTRAP_REPLICATE",
        dtype=object,
    ),
    "replicate": np.arange(1, N_BOOT + 1, dtype=np.int32),
    "sampled_rows": np.full(N_BOOT, EXPECTED_ROWS, dtype=np.int32),
    "seed": np.full(N_BOOT, SEED, dtype=np.int32),
    "rng": np.full(N_BOOT, "numpy.random.Generator", dtype=object),
    "bit_generator": np.full(
        N_BOOT,
        type(rng.bit_generator).__name__,
        dtype=object,
    ),
}

for component_key in COMPONENT_KEYS:
    replicate_data[f"{component_key}__sampled_events"] = sampled_events[component_key]
    replicate_data[f"{component_key}__sampled_negatives"] = (
        EXPECTED_ROWS - sampled_events[component_key]
    )
    replicate_data[f"{component_key}__valid_two_class"] = validity[component_key]

    for model_key in MODEL_KEYS:
        replicate_data[
            f"{component_key}__{model_key}__auprc"
        ] = metric_values[(component_key, model_key, "auprc")]
        replicate_data[
            f"{component_key}__{model_key}__auroc"
        ] = metric_values[(component_key, model_key, "auroc")]

    for fraction in RISK_FRACTIONS:
        fraction_label = f"top_{int(fraction * 100):02d}_percent"
        for metric in (
            "selected_event_rate",
            "risk_ratio_vs_remaining",
            "enrichment_over_prevalence",
        ):
            replicate_data[
                f"{component_key}__{fraction_label}__{metric}"
            ] = enrichment_values[(component_key, fraction, metric)]

replicates = pd.DataFrame(replicate_data)

validity_rows = []
for component_key in COMPONENT_KEYS:
    valid_count = int(validity[component_key].sum())
    invalid_count = N_BOOT - valid_count
    validity_rows.append({
        "analysis_category": "PRIMARY_EVENT_COMPONENT_BOOTSTRAP_VALIDITY",
        "component_key": component_key,
        "event_component_column": COMPONENTS[component_key]["column"],
        "component": COMPONENTS[component_key]["display"],
        "events": COMPONENTS[component_key]["expected_events"],
        "attempted_bootstrap_replicates": N_BOOT,
        "valid_bootstrap_replicates": valid_count,
        "invalid_one_class_replicates": invalid_count,
    })

bootstrap_validity = pd.DataFrame(validity_rows)

interval_rows = []
for component_key in COMPONENT_KEYS:
    prevalence = COMPONENTS[component_key]["expected_events"] / EXPECTED_ROWS
    replicate_prevalence = sampled_events[component_key] / EXPECTED_ROWS

    for model_key in MODEL_KEYS:
        ap_values = metric_values[(component_key, model_key, "auprc")]
        auc_values = metric_values[(component_key, model_key, "auroc")]
        ap_low, ap_high = ci(ap_values)
        auc_low, auc_high = ci(auc_values)
        ap_null_low, ap_null_high = ci(ap_values - replicate_prevalence)
        auc_null_low, auc_null_high = ci(auc_values - 0.50)
        point = point_lookup.loc[(component_key, model_key)]

        interval_rows.append({
            "analysis_category": "PRIMARY_EVENT_COMPONENT_AUPRC_AUROC_BOOTSTRAP_INTERVAL",
            "component_key": component_key,
            "event_component_column": COMPONENTS[component_key]["column"],
            "component": COMPONENTS[component_key]["display"],
            "model_key": model_key,
            "model": MODELS[model_key]["display"],
            "rows": EXPECTED_ROWS,
            "events": COMPONENTS[component_key]["expected_events"],
            "negatives": EXPECTED_ROWS - COMPONENTS[component_key]["expected_events"],
            "prevalence": prevalence,
            "point_auprc": float(point["point_auprc"]),
            "auprc_ci_lower": ap_low,
            "auprc_ci_upper": ap_high,
            "point_auprc_minus_prevalence": float(point["auprc_minus_prevalence"]),
            "auprc_minus_prevalence_ci_lower": ap_null_low,
            "auprc_minus_prevalence_ci_upper": ap_null_high,
            "auprc_null_status": interval_status(
                ap_null_low,
                ap_null_high,
                "supported_above_prevalence",
                "supported_below_prevalence",
            ),
            "point_auroc": float(point["point_auroc"]),
            "auroc_ci_lower": auc_low,
            "auroc_ci_upper": auc_high,
            "point_auroc_minus_0_50": float(point["auroc_minus_0_50"]),
            "auroc_minus_0_50_ci_lower": auc_null_low,
            "auroc_minus_0_50_ci_upper": auc_null_high,
            "auroc_null_status": interval_status(
                auc_null_low,
                auc_null_high,
                "supported_above_0_50",
                "supported_below_0_50",
            ),
            "attempted_bootstrap_replicates": N_BOOT,
            "valid_bootstrap_replicates": int(np.isfinite(ap_values).sum()),
            "invalid_one_class_replicates": int((~np.isfinite(ap_values)).sum()),
        })

intervals = pd.DataFrame(interval_rows)
interval_lookup = intervals.set_index(["component_key", "model_key"])

paired_rows = []

for component_key in COMPONENT_KEYS:
    for comparator_key in PRINCIPAL_COMPARATORS + SECONDARY_COMPARATORS:
        comparison_family = (
            "principal_prespecified"
            if comparator_key in PRINCIPAL_COMPARATORS
            else "secondary_remaining_comparators"
        )

        for metric_label, metric_key in (("AUPRC", "auprc"), ("AUROC", "auroc")):
            differences = (
                metric_values[(component_key, "full_ges", metric_key)]
                - metric_values[(component_key, comparator_key, metric_key)]
            )
            lower, upper = ci(differences)
            point_difference = float(
                point_lookup.loc[(component_key, "full_ges"), f"point_{metric_key}"]
                - point_lookup.loc[(component_key, comparator_key), f"point_{metric_key}"]
            )

            paired_rows.append({
                "analysis_category": "PRIMARY_EVENT_COMPONENT_PAIRED_BOOTSTRAP_INTERVAL",
                "component_key": component_key,
                "event_component_column": COMPONENTS[component_key]["column"],
                "component": COMPONENTS[component_key]["display"],
                "metric": metric_label,
                "comparison_family": comparison_family,
                "comparison": (
                    f"Full GES minus {MODELS[comparator_key]['display']}"
                ),
                "comparator_key": comparator_key,
                "comparator": MODELS[comparator_key]["display"],
                "point_difference": point_difference,
                "difference_ci_lower": lower,
                "difference_ci_upper": upper,
                "paired_interval_status": interval_status(
                    lower,
                    upper,
                    "full_ges_supported_higher",
                    "full_ges_supported_lower",
                ),
                "bootstrap_probability_full_greater": float(
                    np.mean(differences[np.isfinite(differences)] > 0)
                ),
                "bootstrap_sign_p_value": sign_p(differences),
                "secondary_family_holm_adjusted_p": np.nan,
                "secondary_family_holm_supported_at_0_05": pd.NA,
                "attempted_bootstrap_replicates": N_BOOT,
                "valid_bootstrap_replicates": int(np.isfinite(differences).sum()),
                "invalid_one_class_replicates": int((~np.isfinite(differences)).sum()),
            })

paired = pd.DataFrame(paired_rows)

for component_key in COMPONENT_KEYS:
    for metric_label in ("AUPRC", "AUROC"):
        mask = (
            paired["component_key"].eq(component_key)
            & paired["metric"].eq(metric_label)
            & paired["comparison_family"].eq("secondary_remaining_comparators")
        )
        subset = paired.loc[mask].copy()

        if set(subset["comparator_key"]) != set(SECONDARY_COMPARATORS):
            raise RuntimeError("Secondary Holm family membership mismatch.")

        adjusted = holm(subset["bootstrap_sign_p_value"].to_numpy(dtype=float))
        paired.loc[
            subset.index,
            "secondary_family_holm_adjusted_p",
        ] = adjusted
        paired.loc[
            subset.index,
            "secondary_family_holm_supported_at_0_05",
        ] = adjusted <= 0.05

enrichment_rows = []

for component_key in COMPONENT_KEYS:
    outcome = outcomes[component_key]
    prevalence = float(outcome.mean())

    for fraction in RISK_FRACTIONS:
        selected_mask = selected_masks[fraction].astype(bool)
        selected_rows = int(selected_mask.sum())
        selected_events = int(outcome[selected_mask].sum())
        remaining_rows = EXPECTED_ROWS - selected_rows
        remaining_events = int(outcome[~selected_mask].sum())

        selected_rate = selected_events / selected_rows
        remaining_rate = remaining_events / remaining_rows
        point_risk_ratio = (
            selected_rate / remaining_rate
            if remaining_rate > 0
            else np.nan
        )
        point_enrichment = selected_rate / prevalence

        selected_values = enrichment_values[
            (component_key, fraction, "selected_event_rate")
        ]
        ratio_values = enrichment_values[
            (component_key, fraction, "risk_ratio_vs_remaining")
        ]
        enrichment_bootstrap_values = enrichment_values[
            (component_key, fraction, "enrichment_over_prevalence")
        ]

        selected_low, selected_high = ci(selected_values)
        ratio_low, ratio_high = ci(ratio_values)
        enrichment_low, enrichment_high = ci(enrichment_bootstrap_values)

        enrichment_rows.append({
            "analysis_category": "PRIMARY_EVENT_COMPONENT_ENRICHMENT_BOOTSTRAP_INTERVAL",
            "component_key": component_key,
            "event_component_column": COMPONENTS[component_key]["column"],
            "component": COMPONENTS[component_key]["display"],
            "risk_fraction": fraction,
            "selected_rows": selected_rows,
            "selected_events": selected_events,
            "selected_event_rate": selected_rate,
            "selected_event_rate_ci_lower": selected_low,
            "selected_event_rate_ci_upper": selected_high,
            "remaining_rows": remaining_rows,
            "remaining_events": remaining_events,
            "remaining_event_rate": remaining_rate,
            "component_prevalence": prevalence,
            "point_risk_ratio_vs_remaining": point_risk_ratio,
            "risk_ratio_ci_lower": ratio_low,
            "risk_ratio_ci_upper": ratio_high,
            "point_enrichment_over_prevalence": point_enrichment,
            "enrichment_ci_lower": enrichment_low,
            "enrichment_ci_upper": enrichment_high,
            "enrichment_interval_status": interval_status(
                enrichment_low - 1.0,
                enrichment_high - 1.0,
                "supported_above_1",
                "supported_below_1",
            ),
            "attempted_bootstrap_replicates": N_BOOT,
            "valid_enrichment_replicates": int(
                np.isfinite(enrichment_bootstrap_values).sum()
            ),
        })

enrichment = pd.DataFrame(enrichment_rows)
enrichment_lookup = enrichment.set_index(["component_key", "risk_fraction"])


# --------------------------------------------------------------------------------------------------
# 8. HISTORICAL RESULTS AND CONCORDANCE
# --------------------------------------------------------------------------------------------------

historical_rows = []

for component_key, specification in COMPONENTS.items():
    historical_rows.append({
        "result_id": f"{component_key}__events",
        "result_type": "component_accounting",
        "component_key": component_key,
        "metric": "events",
        "risk_fraction": np.nan,
        "historical_value": float(specification["expected_events"]),
        "tolerance": 0.0,
    })

for result_id, value in [
    ("component_or__primary_events", 6485.0),
    ("material_new_conflict__overlap", 6.0),
]:
    historical_rows.append({
        "result_id": result_id,
        "result_type": "component_accounting",
        "component_key": "",
        "metric": result_id,
        "risk_fraction": np.nan,
        "historical_value": value,
        "tolerance": 0.0,
    })

historical_full_ges = {
    "material_group_change": {
        "auprc": (0.025254, 0.023726, 0.026994),
        "auroc": (0.606188, 0.594553, 0.617713),
    },
    "new_unresolved_conflict": {
        "auprc": (0.065003, 0.062902, 0.067273),
        "auroc": (0.481942, 0.474308, 0.489638),
    },
    "material_prior_conflict_resolution": {
        "auprc": (0.232245, 0.199562, 0.276629),
        "auroc": (0.991004, 0.990037, 0.991908),
    },
}

for component_key, metric_spec in historical_full_ges.items():
    for metric, values in metric_spec.items():
        point_value, lower_value, upper_value = values
        for statistic, value in [
            ("point", point_value),
            ("ci_lower", lower_value),
            ("ci_upper", upper_value),
        ]:
            historical_rows.append({
                "result_id": (
                    f"{component_key}__full_ges__{metric}__{statistic}"
                ),
                "result_type": "full_ges_model_interval",
                "component_key": component_key,
                "metric": f"{metric}_{statistic}",
                "risk_fraction": np.nan,
                "historical_value": value,
                "tolerance": 5.1e-7,
            })

historical_enrichment = {
    "new_unresolved_conflict": {
        0.05: (0.258911, 0.194305, 0.327217),
        0.10: (0.386279, 0.332811, 0.439366),
        0.20: (0.763162, 0.712774, 0.812120),
    },
    "material_prior_conflict_resolution": {
        0.05: (19.998800, 19.376422, 20.694571),
        0.10: (9.999400, 9.777807, 10.231269),
        0.20: (4.999700, 4.924328, 5.076255),
    },
}

for component_key, fractions in historical_enrichment.items():
    for fraction, values in fractions.items():
        point_value, lower_value, upper_value = values
        for statistic, value in [
            ("point", point_value),
            ("ci_lower", lower_value),
            ("ci_upper", upper_value),
        ]:
            historical_rows.append({
                "result_id": (
                    f"{component_key}__top_{int(fraction * 100):02d}"
                    f"__enrichment__{statistic}"
                ),
                "result_type": "enrichment_interval",
                "component_key": component_key,
                "metric": f"enrichment_{statistic}",
                "risk_fraction": fraction,
                "historical_value": value,
                "tolerance": 5.1e-7,
            })

for component_key in COMPONENT_KEYS:
    historical_rows.append({
        "result_id": f"{component_key}__valid_replicates",
        "result_type": "bootstrap_validity",
        "component_key": component_key,
        "metric": "valid_replicates",
        "risk_fraction": np.nan,
        "historical_value": 2000.0,
        "tolerance": 0.0,
    })

historical = pd.DataFrame(historical_rows)
historical["analysis_category"] = (
    "PRIMARY_EVENT_COMPONENT_HISTORICAL_BOOTSTRAP_INTERVAL_RESULT"
)
historical["source"] = (
    "Technical report Version 7.0 Appendix O.4 and original Cell 6C-3E1 output."
)

concordance_rows = []

for record in historical.to_dict("records"):
    result_type = record["result_type"]
    component_key = record["component_key"]
    metric = record["metric"]
    reproduced_value = np.nan

    if result_type == "component_accounting":
        if metric == "events":
            reproduced_value = float(COMPONENTS[component_key]["expected_events"])
        elif metric == "component_or__primary_events":
            reproduced_value = float(component_union.sum())
        elif metric == "material_new_conflict__overlap":
            reproduced_value = float(overlap_material_new)
        else:
            raise RuntimeError(f"Unhandled accounting metric: {metric}")

    elif result_type == "full_ges_model_interval":
        row = interval_lookup.loc[(component_key, "full_ges")]
        mapping = {
            "auprc_point": "point_auprc",
            "auprc_ci_lower": "auprc_ci_lower",
            "auprc_ci_upper": "auprc_ci_upper",
            "auroc_point": "point_auroc",
            "auroc_ci_lower": "auroc_ci_lower",
            "auroc_ci_upper": "auroc_ci_upper",
        }
        reproduced_value = float(row[mapping[metric]])

    elif result_type == "enrichment_interval":
        row = enrichment_lookup.loc[
            (component_key, float(record["risk_fraction"]))
        ]
        mapping = {
            "enrichment_point": "point_enrichment_over_prevalence",
            "enrichment_ci_lower": "enrichment_ci_lower",
            "enrichment_ci_upper": "enrichment_ci_upper",
        }
        reproduced_value = float(row[mapping[metric]])

    elif result_type == "bootstrap_validity":
        reproduced_value = float(
            bootstrap_validity.loc[
                bootstrap_validity["component_key"].eq(component_key),
                "valid_bootstrap_replicates",
            ].iloc[0]
        )

    else:
        raise RuntimeError(f"Unhandled historical result type: {result_type}")

    absolute_difference = abs(
        reproduced_value - float(record["historical_value"])
    )
    concordance_rows.append({
        **record,
        "reproduced_value": reproduced_value,
        "absolute_difference": absolute_difference,
        "reproduced_at_recorded_precision": (
            absolute_difference <= float(record["tolerance"])
        ),
    })

concordance = pd.DataFrame(concordance_rows)


# --------------------------------------------------------------------------------------------------
# 9. SCIENTIFIC-CONCLUSION AUDIT AND QC
# --------------------------------------------------------------------------------------------------

material_full = interval_lookup.loc[("material_group_change", "full_ges")]
new_conflict_full = interval_lookup.loc[("new_unresolved_conflict", "full_ges")]
resolution_full = interval_lookup.loc[
    ("material_prior_conflict_resolution", "full_ges")
]

scientific_conclusions = {
    "component_or_reconstructs_primary": bool(
        np.array_equal(component_union, primary)
    ),
    "material_group_change_full_above_both_nulls": bool(
        material_full["auprc_null_status"] == "supported_above_prevalence"
        and material_full["auroc_null_status"] == "supported_above_0_50"
    ),
    "new_conflict_full_below_both_nulls": bool(
        new_conflict_full["auprc_null_status"] == "supported_below_prevalence"
        and new_conflict_full["auroc_null_status"] == "supported_below_0_50"
    ),
    "new_conflict_enrichment_below_one_all_fractions": bool(
        (
            enrichment.loc[
                enrichment["component_key"].eq("new_unresolved_conflict"),
                "enrichment_interval_status",
            ]
            == "supported_below_1"
        ).all()
    ),
    "prior_resolution_full_above_both_nulls": bool(
        resolution_full["auprc_null_status"] == "supported_above_prevalence"
        and resolution_full["auroc_null_status"] == "supported_above_0_50"
    ),
    "prior_resolution_all_events_in_top_5_percent": bool(
        int(
            enrichment_lookup.loc[
                ("material_prior_conflict_resolution", 0.05),
                "selected_events",
            ]
        )
        == 297
    ),
    "combined_metadata_exceeds_full_for_prior_resolution": bool(
        point_lookup.loc[
            ("material_prior_conflict_resolution", "combined_metadata"),
            "point_auprc",
        ]
        > point_lookup.loc[
            ("material_prior_conflict_resolution", "full_ges"),
            "point_auprc",
        ]
        and point_lookup.loc[
            ("material_prior_conflict_resolution", "combined_metadata"),
            "point_auroc",
        ]
        > point_lookup.loc[
            ("material_prior_conflict_resolution", "full_ges"),
            "point_auroc",
        ]
    ),
    "all_components_have_2000_valid_replicates": bool(
        (
            bootstrap_validity["valid_bootstrap_replicates"]
            == N_BOOT
        ).all()
        and (
            bootstrap_validity["invalid_one_class_replicates"]
            == 0
        ).all()
    ),
}

checks = []

def check(name: str, passed: bool, details):
    checks.append({
        "check_name": name,
        "passed": bool(passed),
        "details": native(details),
    })

check("source_hash", sha(SOURCE) == SOURCE_SHA256, sha(SOURCE))
check("source_sidecar", sidecar_ok(SOURCE), str(SOURCE) + ".sha256")
check(
    "prior_4k0a_manifest_hash",
    sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA256,
    sha(PRIOR_MANIFEST),
)
check(
    "prior_4k0a_manifest_sidecar",
    sidecar_ok(PRIOR_MANIFEST),
    str(PRIOR_MANIFEST) + ".sha256",
)
check(
    "source_dimensions",
    (metadata.num_rows, metadata.num_columns) == (
        EXPECTED_ROWS,
        EXPECTED_COLUMNS,
    ),
    {
        "rows": metadata.num_rows,
        "columns": metadata.num_columns,
    },
)
check(
    "primary_accounting",
    (int(primary.sum()), int((primary == 0).sum()))
    == (EXPECTED_PRIMARY_EVENTS, EXPECTED_PRIMARY_NEGATIVES),
    {},
)
check(
    "component_event_counts",
    accounting["component_event_count_matches"].all(),
    accounting.to_dict("records"),
)
check(
    "component_overlap_accounting",
    (
        overlap_material_new,
        overlap_material_resolution,
        overlap_new_resolution,
        triple_overlap,
    )
    == (6, 0, 0, 0),
    {},
)
check(
    "component_union_reconstruction",
    scientific_conclusions["component_or_reconstructs_primary"],
    int(component_union.sum()),
)
check("nine_scores", len(score_arrays) == 9, list(score_arrays))
check(
    "exact_grouped_metric_validation_27",
    len(metric_validation) == 27
    and (
        metric_validation[
            ["auprc_absolute_difference", "auroc_absolute_difference"]
        ].to_numpy(float)
        <= 1e-12
    ).all(),
    metric_validation.to_dict("records"),
)
check("point_estimate_rows", len(points) == 27, len(points))
check("bootstrap_replicate_rows", len(replicates) == N_BOOT, len(replicates))
check(
    "bootstrap_validity",
    scientific_conclusions["all_components_have_2000_valid_replicates"],
    bootstrap_validity.to_dict("records"),
)
check("model_interval_rows", len(intervals) == 27, len(intervals))
check("paired_inference_rows", len(paired) == 48, len(paired))
check(
    "secondary_holm_complete",
    paired.loc[
        paired["comparison_family"].eq("secondary_remaining_comparators"),
        "secondary_family_holm_adjusted_p",
    ].notna().all(),
    {},
)
check(
    "principal_not_in_secondary_holm",
    paired.loc[
        paired["comparison_family"].eq("principal_prespecified"),
        "secondary_family_holm_adjusted_p",
    ].isna().all(),
    {},
)
check("enrichment_rows", len(enrichment) == 9, len(enrichment))
check(
    "historical_concordance",
    concordance["reproduced_at_recorded_precision"].all(),
    concordance.loc[
        ~concordance["reproduced_at_recorded_precision"],
        [
            "result_id",
            "historical_value",
            "reproduced_value",
            "absolute_difference",
            "tolerance",
        ],
    ].to_dict("records"),
)

for conclusion_name, passed in scientific_conclusions.items():
    check(
        f"scientific_conclusion__{conclusion_name}",
        passed,
        passed,
    )

check(
    "frozen_sources_unchanged",
    sha(SOURCE) == SOURCE_SHA256
    and sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA256,
    {},
)

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError(
        "QC failed before writing:\n"
        + json.dumps(native(failed), indent=2)
    )


# --------------------------------------------------------------------------------------------------
# 10. VERSIONED WRITES, SIDECARS, MANIFEST, AND FRESH READBACK
# --------------------------------------------------------------------------------------------------

write_csv(P["accounting"], accounting)
write_csv(P["points"], points)
write_parquet(P["replicates"], replicates)
write_csv(P["validity"], bootstrap_validity)
write_csv(P["intervals"], intervals)
write_csv(P["paired"], paired)
write_csv(P["enrichment"], enrichment)
write_csv(P["historical"], historical)
write_csv(P["concordance"], concordance)

readback = {
    "accounting": len(pd.read_csv(P["accounting"])) == 3,
    "points": len(pd.read_csv(P["points"])) == 27,
    "replicates": len(pd.read_parquet(P["replicates"])) == N_BOOT,
    "validity": len(pd.read_csv(P["validity"])) == 3,
    "intervals": len(pd.read_csv(P["intervals"])) == 27,
    "paired": len(pd.read_csv(P["paired"])) == 48,
    "enrichment": len(pd.read_csv(P["enrichment"])) == 9,
    "historical": len(pd.read_csv(P["historical"])) == len(historical),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
}

if not all(readback.values()):
    raise RuntimeError(f"Artifact readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4K0B",
    "package_version": "v1",
    "created_utc": CREATED_UTC,
    "analysis_category": "PRIMARY_EVENT_COMPONENT_BOOTSTRAP_INTERVAL_SUPPORT",
    "source": {
        "path": str(SOURCE),
        "sha256": sha(SOURCE),
    },
    "prior_manifest": {
        "path": str(PRIOR_MANIFEST),
        "sha256": sha(PRIOR_MANIFEST),
    },
    "bootstrap": {
        "method": "paired ordinary multinomial row-count bootstrap",
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
        "seed": SEED,
        "attempts": N_BOOT,
        "batch_size": BOOTSTRAP_BATCH_SIZE,
        "identical_resamples_across_components_and_scores": True,
        "validity": bootstrap_validity.to_dict("records"),
    },
    "checks": checks,
    "table_readback": readback,
    "passed_checks": sum(item["passed"] for item in checks),
    "failed_checks": sum(not item["passed"] for item in checks),
    "decision": (
        "PASS_STAGE6C_PRIMARY_EVENT_COMPONENT_BOOTSTRAP_SUPPORT_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)

artifact_keys = [
    "accounting",
    "points",
    "replicates",
    "validity",
    "intervals",
    "paired",
    "enrichment",
    "historical",
    "concordance",
    "qc",
]

for key in artifact_keys:
    sidecar(P[key])
    if not sidecar_ok(P[key]):
        raise RuntimeError(f"Sidecar verification failed: {P[key]}")


def artifact_record(key: str) -> dict:
    path = P[key]
    record = {
        "artifact_key": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path),
        "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name + ".sha256")),
        "sidecar_verified": sidecar_ok(path),
    }

    if path.suffix == ".csv":
        loaded = pd.read_csv(path)
        record.update(rows=len(loaded), columns=loaded.shape[1])
    elif path.suffix == ".parquet":
        loaded_metadata = pq.ParquetFile(path).metadata
        record.update(
            rows=loaded_metadata.num_rows,
            columns=loaded_metadata.num_columns,
        )
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True

    return record


manifest = {
    "cell_id": "6C-4K0B",
    "package_version": "v1",
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": CREATED_UTC,
    "authorized_category": "primary_event_component_bootstrap_support_materialization",
    "analysis_category": "PRIMARY_EVENT_COMPONENT_AUPRC_AUROC_BOOTSTRAP_INTERVAL",
    "immutable_sources": {
        "stage6b_primary_evaluable": {
            "path": str(SOURCE),
            "sha256": sha(SOURCE),
            "expected_sha256": SOURCE_SHA256,
        },
        "prior_4k0a_manifest": {
            "path": str(PRIOR_MANIFEST),
            "sha256": sha(PRIOR_MANIFEST),
            "expected_sha256": PRIOR_MANIFEST_SHA256,
        },
    },
    "analysis_lock": {
        "components": COMPONENTS,
        "scores": MODELS,
        "primary_comparators": PRINCIPAL_COMPARATORS,
        "secondary_comparators": SECONDARY_COMPARATORS,
        "risk_fractions": RISK_FRACTIONS,
        "bootstrap_method": "multinomial ordinary row-count bootstrap",
        "bootstrap_seed": SEED,
        "bootstrap_attempts": N_BOOT,
        "bootstrap_batch_size": BOOTSTRAP_BATCH_SIZE,
        "secondary_holm_policy": (
            "Holm across five remaining comparators separately "
            "within each event component and metric."
        ),
    },
    "accounting": {
        "primary_events": int(primary.sum()),
        "material_group_change_events": int(material.sum()),
        "new_unresolved_conflict_events": int(new_conflict.sum()),
        "material_prior_conflict_resolution_events": int(prior_resolution.sum()),
        "material_and_new_conflict_overlap": overlap_material_new,
        "other_pairwise_overlaps": 0,
        "triple_overlap": triple_overlap,
        "component_union_events": int(component_union.sum()),
    },
    "result_summary": {
        "material_group_change_full_ges_auprc": float(
            material_full["point_auprc"]
        ),
        "material_group_change_full_ges_auroc": float(
            material_full["point_auroc"]
        ),
        "new_unresolved_conflict_full_ges_auprc": float(
            new_conflict_full["point_auprc"]
        ),
        "new_unresolved_conflict_full_ges_auroc": float(
            new_conflict_full["point_auroc"]
        ),
        "prior_conflict_resolution_full_ges_auprc": float(
            resolution_full["point_auprc"]
        ),
        "prior_conflict_resolution_full_ges_auroc": float(
            resolution_full["point_auroc"]
        ),
        "scientific_conclusion_audit": scientific_conclusions,
        "historical_results_concordant": bool(
            concordance["reproduced_at_recorded_precision"].all()
        ),
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "artifacts": [artifact_record(key) for key in artifact_keys],
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "scores_refit_or_recalibrated": False,
        "thresholds_or_weights_changed": False,
        "component_definitions_changed": False,
        "primary_outcome_changed": False,
        "cohort_membership_changed": False,
        "experiment_2_started": False,
        "interpretation": (
            "The frozen primary outcome is strongly component-dependent. "
            "Full GES shows modest positive ranking for material clinical-group change, "
            "below-null performance for newly emerging unresolved conflict, and extremely "
            "strong ranking for material prior-conflict resolution. These findings do not "
            "support uniform future-instability prediction."
        ),
    },
    "decision": (
        "PASS_STAGE6C_PRIMARY_EVENT_COMPONENT_BOOTSTRAP_SUPPORT_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "next_authorized_action": (
        "rerun_cell_6c_4k0_final_integrated_package_freeze_final_corrected_v2"
    ),
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

manifest_readback = json.loads(P["manifest"].read_text(encoding="utf-8"))

if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Manifest sidecar verification failed.")
if manifest_readback["decision"] != manifest["decision"]:
    raise RuntimeError("Manifest decision readback mismatch.")
if manifest_readback["scientific_boundary"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed.")

for artifact in manifest_readback["artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final artifact verification failed: {path}")

if sha(SOURCE) != SOURCE_SHA256 or sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256:
    raise RuntimeError("An immutable source changed during Cell 6C-4K0B.")


# --------------------------------------------------------------------------------------------------
# 11. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 170)
print(
    "STAGE 6C — CELL 6C-4K0B — "
    "PRIMARY EVENT-COMPONENT BOOTSTRAP SUPPORT MATERIALIZATION"
)
print("=" * 170)
print(f"Notebook file name                  : {NOTEBOOK_NAME}")
print(f"Stage 6B source SHA-256             : PASS ({sha(SOURCE)})")
print(f"Prior Cell 6C-4K0A manifest         : PASS ({sha(PRIOR_MANIFEST)})")
print(
    f"Frozen cohort                       : PASS "
    f"({EXPECTED_ROWS:,} rows; {EXPECTED_PRIMARY_EVENTS:,} events; "
    f"{EXPECTED_PRIMARY_NEGATIVES:,} negatives)"
)
print(
    "Component accounting               : PASS "
    "(1,405 material; 4,789 new conflict; 297 prior resolution; "
    "six material+new-conflict overlaps)"
)
print("Exact metric validation             : PASS (27/27)")
print(f"Bootstrap design                    : PASS ({N_BOOT:,} paired replicates; seed {SEED}; batch {BOOTSTRAP_BATCH_SIZE})")
print(
    f"Valid / invalid component replicates: "
    f"{int(bootstrap_validity.valid_bootstrap_replicates.sum()):,} / "
    f"{int(bootstrap_validity.invalid_one_class_replicates.sum()):,}"
)
print(f"Model interval rows                 : PASS ({len(intervals)}/27)")
print(f"Paired comparison rows              : PASS ({len(paired)}/48)")
print(f"Enrichment interval rows            : PASS ({len(enrichment)}/9)")
print(
    f"Historical concordance              : PASS "
    f"({int(concordance.reproduced_at_recorded_precision.sum())}/{len(concordance)})"
)
print(f"Fresh QC                            : PASS ({qc_payload['passed_checks']}/{len(checks)})")
print(f"Bootstrap elapsed                   : {bootstrap_elapsed:.1f}s")
print(f"Output table directory              : {TABLE_DIR}")
print(f"QC path                             : {P['qc']}")
print(f"Manifest path                       : {P['manifest']}")
print(f"Manifest SHA-256                    : PASS ({manifest_hash})")

print("\nFULL-GES EVENT-COMPONENT INTERVALS")
print(
    intervals.loc[
        intervals["model_key"].eq("full_ges"),
        [
            "component",
            "events",
            "prevalence",
            "point_auprc",
            "auprc_ci_lower",
            "auprc_ci_upper",
            "auprc_null_status",
            "point_auroc",
            "auroc_ci_lower",
            "auroc_ci_upper",
            "auroc_null_status",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ],
    ].to_string(index=False)
)

print("\nNEW-CONFLICT AND PRIOR-RESOLUTION FULL-GES ENRICHMENT")
print(
    enrichment.loc[
        enrichment["component_key"].isin(
            [
                "new_unresolved_conflict",
                "material_prior_conflict_resolution",
            ]
        ),
        [
            "component",
            "risk_fraction",
            "selected_rows",
            "selected_events",
            "point_enrichment_over_prevalence",
            "enrichment_ci_lower",
            "enrichment_ci_upper",
            "enrichment_interval_status",
        ],
    ].to_string(index=False)
)

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print(
    "The global primary result is component-dependent. Full GES shows modest "
    "positive ranking for material clinical-group change, below-null ranking and "
    "top-risk depletion for newly emerging unresolved conflict, and extremely strong "
    "ranking for material prior-conflict resolution. This cell only serializes the "
    "already-completed Cell 6C-3E1 analysis; no score, outcome, threshold, weight, "
    "linkage decision, row order, or cohort membership was changed."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_PRIMARY_EVENT_COMPONENT_BOOTSTRAP_SUPPORT_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)
print(
    "The previously in-memory event-component bootstrap is now independently "
    "serialized and checksum-protected. Next authorized action: rerun Cell 6C-4K0 "
    "FINAL_CORRECTED_V2 from the beginning. Experiment 2 remains unstarted."
)
print("=" * 170)


Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4K0B_Primary_Event_Component_Bootstrap_Support_Materialization.ipynb

Reconstructing primary event-component bootstrap across 66,636 rows: Material clinical-group change=1,405, New unresolved conflict=4,789, Material prior-conflict resolution=297
Exact grouped metric validation against scikit-learn: PASS (27/27)
  Completed 250/2,000 replicates | valid [material_group_change=250, new_unresolved_conflict=250, material_prior_conflict_resolution=250]
  Completed 500/2,000 replicates | valid [material_group_change=500, new_unresolved_conflict=500, material_prior_conflict_resolution=500]
  Completed 750/2,000 replicates | valid [material_group_change=750, new_unresolved_conflict=750, material_prior_conflict_resolution=750]
  Completed 1,000/2,000 replicates | valid [material_group_change=1,000, new_unresolved_conflict=1,000, material_prior_conflict_resolution=1,000]
  Completed 1,250/2,000 replicates | valid [m